# 01 · Data exploration

First look at the raw US Pollution data, before writing any cleaning code.

Questions:
1. What does the data look like, and how much memory does it use?
2. Why do some station-days appear on several rows?
3. How many values are missing?
4. Does each pollutant use a single unit?
5. Are there impossible values (e.g. negatives)?
6. How many stations are there, and over what period?

In [13]:
import pandas as pd

from applied_data_science_ri import config

pd.set_option("display.max_columns", None)

# Use a sample while exploring; set NROWS = None to load the full file (~1.7M rows).
# The file is sorted by date, so the sample only covers the first couple of years.
NROWS = None

df = pd.read_csv(config.RAW_DIR / config.RAW_FILENAME, nrows=NROWS)
df.shape

(1746661, 29)

## 1. What the data looks like

In [14]:
df.head()

,Unnamed: 0,State Code,County Code,Site Num,Address,State,County,City,Date Local,NO2 Units,NO2 Mean,NO2 1st Max Value,NO2 1st Max Hour,NO2 AQI,O3 Units,O3 Mean,O3 1st Max Value,O3 1st Max Hour,O3 AQI,SO2 Units,SO2 Mean,SO2 1st Max Value,SO2 1st Max Hour,SO2 AQI,CO Units,CO Mean,CO 1st Max Value,CO 1st Max Hour,CO AQI
0,0,4,13,3002,1645 E ROOSEVELT ST-CENTRAL PHOENIX STN,Arizona,Maricopa,Phoenix,2000-01-01,Parts per billion,19.041667,49.0,19,46,Parts per million,0.022500,0.040,10,34,Parts per billion,3.000000,9.0,21,13.0,Parts per million,1.145833,4.2,21,NaN
1,1,4,13,3002,1645 E ROOSEVELT ST-CENTRAL PHOENIX STN,Arizona,Maricopa,Phoenix,2000-01-01,Parts per billion,19.041667,49.0,19,46,Parts per million,0.022500,0.040,10,34,Parts per billion,3.000000,9.0,21,13.0,Parts per million,0.878947,2.2,23,25.0
2,2,4,13,3002,1645 E ROOSEVELT ST-CENTRAL PHOENIX STN,Arizona,Maricopa,Phoenix,2000-01-01,Parts per billion,19.041667,49.0,19,46,Parts per million,0.022500,0.040,10,34,Parts per billion,2.975000,6.6,23,NaN,Parts per million,1.145833,4.2,21,NaN
3,3,4,13,3002,1645 E ROOSEVELT ST-CENTRAL PHOENIX STN,Arizona,Maricopa,Phoenix,2000-01-01,Parts per billion,19.041667,49.0,19,46,Parts per million,0.022500,0.040,10,34,Parts per billion,2.975000,6.6,23,NaN,Parts per million,0.878947,2.2,23,25.0
4,4,4,13,3002,1645 E ROOSEVELT ST-CENTRAL PHOENIX STN,Arizona,Maricopa,Phoenix,2000-01-02,Parts per billion,22.958333,36.0,19,34,Parts per million,0.013375,0.032,10,27,Parts per billion,1.958333,3.0,22,4.0,Parts per million,0.850000,1.6,23,NaN


In [15]:
df.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 1746661 entries, 0 to 1746660
Data columns (total 29 columns):
 #   Column             Dtype  
---  ------             -----  
 0   Unnamed: 0         int64  
 1   State Code         int64  
 2   County Code        int64  
 3   Site Num           int64  
 4   Address            str    
 5   State              str    
 6   County             str    
 7   City               str    
 8   Date Local         str    
 9   NO2 Units          str    
 10  NO2 Mean           float64
 11  NO2 1st Max Value  float64
 12  NO2 1st Max Hour   int64  
 13  NO2 AQI            int64  
 14  O3 Units           str    
 15  O3 Mean            float64
 16  O3 1st Max Value   float64
 17  O3 1st Max Hour    int64  
 18  O3 AQI             int64  
 19  SO2 Units          str    
 20  SO2 Mean           float64
 21  SO2 1st Max Value  float64
 22  SO2 1st Max Hour   int64  
 23  SO2 AQI            float64
 24  CO Units           str    
 25  CO Mean            float64
 2

The first column is just the old row index, so we can drop it. Text columns (State, County, units...) are stored as `str`, which takes a lot of memory: most of them repeat a few values, so `category` would be much lighter.

## 2. Duplicated station-days

In [16]:
station_day = ["State Code", "County Code", "Site Num", "Date Local"]

rows_per_station_day = df.groupby(station_day).size()
rows_per_station_day.value_counts()

4      395829
8       12452
16       3623
2         743
12        115
32         65
1          17
6           9
324         2
96          1
Name: count, dtype: int64

In [17]:
# Look at one station-day that appears more than once
example = rows_per_station_day[rows_per_station_day > 1].index[0]
df.set_index(station_day).sort_index().loc[example]

Unnamed: 0  \
State Code County Code Site Num Date Local               
1          73          23       2013-12-01           0   
                                2013-12-01           1   
                                2013-12-01           2   
                                2013-12-01           3   

                                                                          Address  \
State Code County Code Site Num Date Local                                          
1          73          23       2013-12-01  NO. B'HAM,SOU R.R., 3009 28TH ST. NO.   
                                2013-12-01  NO. B'HAM,SOU R.R., 3009 28TH ST. NO.   
                                2013-12-01  NO. B'HAM,SOU R.R., 3009 28TH ST. NO.   
                                2013-12-01  NO. B'HAM,SOU R.R., 3009 28TH ST. NO.   

                                              State     County        City  \
State Code County Code Site Num Date Local                                   
1          73          23       2013-12-01  Alabama  Jefferson  Birmingham   
                                2013-12-01  Alabama  Jefferson  Birmingham   
                                2013-12-01  Alabama  Jefferson  Birmingham   
                                2013-12-01  Alabama  Jefferson  Birmingham   

                                                    NO2 Units   NO2 Mean  \
State Code County Code Site Num Date Local                                 
1          73          23       2013-12-01  Parts per billion  17.208333   
                                2013-12-01  Parts per billion  17.208333   
                                2013-12-01  Parts per billion  17.208333   
                                2013-12-01  Parts per billion  17.208333   

                                            NO2 1st Max Value  \
State Code County Code Site Num Date Local                      
1          73          23       2013-12-01               39.3   
                                2013-12-01               39.3   
                                2013-12-01               39.3   
                                2013-12-01               39.3   

                                            NO2 1st Max Hour  NO2 AQI  \
State Code County Code Site Num Date Local                              
1          73          23       2013-12-01                18       37   
                                2013-12-01                18       37   
                                2013-12-01                18       37   
                                2013-12-01                18       37   

                                                     O3 Units   O3 Mean  \
State Code County Code Site Num Date Local                                
1          73          23       2013-12-01  Parts per million  0.013542   
                                2013-12-01  Parts per million  0.013542   
                                2013-12-01  Parts per million  0.013542   
                                2013-12-01  Parts per million  0.013542   

                                            O3 1st Max Value  O3 1st Max Hour  \
State Code County Code Site Num Date Local                                      
1          73          23       2013-12-01             0.026                9   
                                2013-12-01             0.026                9   
                                2013-12-01             0.026                9   
                                2013-12-01             0.026                9   

                                            O3 AQI          SO2 Units  \
State Code County Code Site Num Date Local                              
1          73          23       2013-12-01      24  Parts per billion   
                                2013-12-01      24  Parts per billion   
                                2013-12-01      24  Parts per billion   
                                2013-12-01      24  Parts per billion   

                                            SO2 Mea

In [18]:
# Which columns actually change between the duplicated rows?
n_unique = df.groupby(station_day).nunique()
(n_unique > 1).mean().sort_values(ascending=False).head(10)

Unnamed: 0           0.999959
CO Mean              0.922593
SO2 Mean             0.890805
SO2 1st Max Hour     0.835836
SO2 1st Max Value    0.785414
CO 1st Max Value     0.774895
CO 1st Max Hour      0.761546
CO AQI               0.018585
SO2 AQI              0.008901
O3 Mean              0.007913
dtype: float64

**Note:** write down which columns differ. That tells us how to collapse the duplicates into one row per station and day.

## 3. Missing values

In [19]:
missing = df.isna().mean().mul(100).round(1)
missing[missing > 0].sort_values(ascending=False)

SO2 AQI    50.0
CO AQI     50.0
dtype: float64

## 4. Units

In [20]:
unit_cols = [c for c in df.columns if c.endswith("Units")]
for col in unit_cols:
    print(col, "->", df[col].unique())

NO2 Units -> <ArrowStringArray>
['Parts per billion']
Length: 1, dtype: str
O3 Units -> <ArrowStringArray>
['Parts per million']
Length: 1, dtype: str
SO2 Units -> <ArrowStringArray>
['Parts per billion']
Length: 1, dtype: str
CO Units -> <ArrowStringArray>
['Parts per million']
Length: 1, dtype: str


## 5. Impossible values

In [21]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Unnamed: 0,1746661.0,54714.136753,33729.076302,0.0000,25753.000000,53045.000000,80336.000000,134575.000000
State Code,1746661.0,22.309065,17.256205,1.0000,6.000000,17.000000,40.000000,80.000000
County Code,1746661.0,71.693807,79.480231,1.0000,17.000000,59.000000,97.000000,650.000000
Site Num,1746661.0,1118.214373,2003.103069,1.0000,9.000000,60.000000,1039.000000,9997.000000
NO2 Mean,1746661.0,12.821930,9.504814,-2.0000,5.750000,10.739130,17.713636,139.541667
NO2 1st Max Value,1746661.0,25.414848,15.999630,-2.0000,13.000000,24.000000,35.700000,267.000000
NO2 1st Max Hour,1746661.0,11.731023,7.877501,0.0000,5.000000,9.000000,20.000000,23.000000
NO2 AQI,1746661.0,23.898217,15.162805,0.0000,12.000000,23.000000,33.000000,132.000000
O3 Mean,1746661.0,0.026125,0.011370,0.0000,0.017875,0.025875,0.033917,0.095083
O3 1st Max Value,1746661.0,0.039203,0.015344,0.0000,0.029000,0.038000,0.048000,0.141000


In [22]:
value_cols = [c for c in df.columns if c.endswith(("Mean", "Max Value", "AQI"))]
(df[value_cols] < 0).sum()

NO2 Mean               896
NO2 1st Max Value      184
NO2 AQI                  0
O3 Mean                  0
O3 1st Max Value         0
O3 AQI                   0
SO2 Mean             22690
SO2 1st Max Value     8286
SO2 AQI                  0
CO Mean               1064
CO 1st Max Value       198
CO AQI                   0
dtype: int64

## 6. Stations and time span

In [23]:
df["Date Local"] = pd.to_datetime(df["Date Local"])

print("Stations:", df.groupby(["State Code", "County Code", "Site Num"]).ngroups)
print("States:", df["State"].nunique())
print("From", df["Date Local"].min().date(), "to", df["Date Local"].max().date())

Stations: 204
States: 47
From 2000-01-01 to 2016-05-31


In [24]:
# Number of days with data per station
days_per_station = df.groupby(["State Code", "County Code", "Site Num"])["Date Local"].nunique()
days_per_station.describe()

count     204.000000
mean     2023.803922
std      1686.533081
min        26.000000
25%       603.750000
50%      1598.500000
75%      3034.500000
max      5922.000000
Name: Date Local, dtype: float64

## 7. Two follow-up checks

In [ ]:
# Are all "states" really US states?
sorted(df["State"].unique())

In [ ]:
# Which stations have the very large duplicate groups (32+ rows per day)?
big = rows_per_station_day[rows_per_station_day >= 32].reset_index()
big.groupby(["State Code", "County Code", "Site Num"]).size()

## Summary

Based on the full file (`NROWS = None`): 1,746,661 rows × 29 columns.

- **Memory:** 607 MB. Most of it is text columns (`Address`, `State`, `County`, `City`, the four `Units` columns) that repeat a few values: load them as `category`. `Unnamed: 0` is not a useful ID and can be dropped. `Date Local` is read as text and must be parsed as a date.
- **Duplicates:** the 1.7M rows correspond to only ~413,000 station-days. Most station-days appear on **4 rows** (395,829), some on 8 or 16, a few on up to 324. NO2 and O3 are (almost always) identical across the copies; what changes is **SO2** and **CO** (mean, max value, max hour). Each station reports SO2 and CO with more than one measurement method, and the file combines every pair (2 SO2 × 2 CO = 4 rows). To get one row per station-day we must decide how to combine them (e.g. average, or keep one method). The groups with 32+ rows come from just 3 stations (Rubidoux CA, Northbrook IL, Boston MA) that use even more SO2/CO methods: averaging handles them too.
- **Missing values:** only `SO2 AQI` and `CO AQI`, each missing in exactly **50%** of rows. This comes from the duplication: the AQI is filled in for only one of the methods. After collapsing the duplicates, keep the AQI value that is present and most gaps should disappear.
- **Units:** each pollutant uses a single unit (NO2 and SO2 in ppb, O3 and CO in ppm). No conversion needed, so the `Units` columns can be dropped.
- **Impossible values:** there **are negative concentrations**: SO2 Mean (22,690 rows), CO Mean (1,064), NO2 Mean (896), plus some negative max values. They are small (down to -2 ppb, -0.44 ppm), probably instrument noise around zero: clip them to 0 or set them to missing. AQI columns have no negatives. Some maximums are extreme (SO2 Mean 321 ppb, CO 1st Max 19.9 ppm) and should be checked as possible outliers.
- **Stations / time span:** 204 stations in 47 "states". The list includes "Country Of Mexico" (not US: drop it) and "District Of Columbia" (keep it), from 2000-01-01 to 2016-05-31. Coverage is very uneven: 26 to 5,922 days per station (median ~1,600 out of ~6,000 possible), so most stations only report for part of the period or have gaps. This matters for lag features: "7 days ago" must mean 7 calendar days, not 7 rows.